In [ ]:
# === ノートブック共通の前処理 (llm_math パッケージの読み込み) ===
import sys
from pathlib import Path

# llm_math パッケージの候補パス
_candidates = [
    '.', 'src', '..', '../src',
    '/content/llm-math-book/src',
    '/content/llm-math-book',
    '/workspace/src',
    '/workspace',
]
# 親ディレクトリも候補に追加 (notebooks/ フォルダで実行する場合)
for p in Path.cwd().parents:
    _candidates.append(str(p / 'src'))
    _candidates.append(str(p))

for p in _candidates:
    if p and p not in sys.path and Path(p).exists():
        sys.path.insert(0, p)

# llm_math の import を試行
try:
    from llm_math import viz, bench, data
    _LLM_MATH_OK = True
except ImportError as e:
    _LLM_MATH_OK = False
    print(f"[注意] llm_math パッケージの読み込み テキスト: {e}")
    print("  GitHub リポジトリを clone して colab_setup.sh を実行してください。")
# === 前処理ここまで ===


# Ch 14. Attention テキスト テキスト — テキスト·テキスト·値 ⭐

> **学習目標**
> - Q, K, Vテキスト テキスト テキスト(database lookup) テキスト テキスト
> - Scaled Dot-Product Attentionテキスト $\sqrt{d_k}$テキスト テキスト テキスト テキスト度テキスト
> - Attentionテキスト 時間/空間 テキスト度 $O(n^2 d)$テキスト テキスト
> - Attention テキスト CPU vs GPUテキスト テキスト 比較テキスト

## 14.1 Attentionテキスト テキスト テキスト

RNNテキスト テキスト: テキスト テキスト テキスト テキスト テキスト テキスト テキスト テキスト. テキスト テキスト テキスト テキスト.

Bahdanau Attention(2014): テキスト テキスト テキスト テキスト **テキスト** テキスト テキスト テキスト.

**Self-Attention** (テキスト): テキスト テキスト テキスト テキスト テキスト テキスト テキスト. RNN テキスト度 テキスト テキスト 学習.

## 14.2 テキスト·テキスト·値 (Q, K, V) — テキスト テキスト

データテキスト テキスト テキスト:
- **Query (Q)**: テキスト テキスト テキスト テキスト
- **Key (K)**: テキスト テキスト テキスト (テキスト テキスト テキスト テキスト テキスト)
- **Value (V)**: テキスト テキスト テキスト テキスト

Attention: Qテキスト テキスト Kテキスト テキスト度テキスト 計算 → テキスト度テキスト テキスト V テキスト.

テキスト:
$$\mathrm{Attention}(Q, K, V) = \mathrm{softmax}\left(\frac{Q K^\top}{\sqrt{d_k}}\right) V$$

- $Q \in \mathbb{R}^{n \times d_k}$: $n$テキスト テキスト
- $K \in \mathbb{R}^{n \times d_k}$: $n$テキスト テキスト
- $V \in \mathbb{R}^{n \times d_v}$: $n$テキスト 値
- $QK^\top \in \mathbb{R}^{n \times n}$: テキスト テキスト-テキスト テキスト テキスト


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F

# Scaled Dot-Product Attention テキスト (NumPy)
def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)
    e = np.exp(x)
    return e / e.sum(axis=axis, keepdims=True)

def attention(Q, K, V, mask=None):
    """Scaled Dot-Product Attention.
    Q, K, V: (n, d) or (batch, n, d)
    """
    d_k = Q.shape[-1]
    scores = Q @ np.swapaxes(K, -1, -2) / np.sqrt(d_k)  # (..., n, n)
    if mask is not None:
        scores = np.where(mask, scores, -1e9)
    weights = softmax(scores, axis=-1)
    output = weights @ V
    return output, weights

# テキスト テキスト
np.random.seed(0)
n, d = 4, 8
Q = np.random.randn(n, d)
K = np.random.randn(n, d)
V = np.random.randn(n, d)

output, weights = attention(Q, K, V)
print(f"Q shape: {Q.shape}")
print(f"Attention output: {output.shape}")
print(f"Attention weights: {weights.shape}")
print(f"\nテキスト テキスト Sum (softmax テキスト): {weights.sum(axis=-1).round(4)}")


## 14.3 テキスト $\sqrt{d_k}$テキスト テキスト?

$QK^\top$テキスト テキスト $\sum_{i=1}^{d_k} Q_i K_i$. Q, Kテキスト $\mathcal{N}(0, 1)$テキスト:

$$\mathrm{Var}(QK^\top_{ij}) = \sum_{i=1}^{d_k} \mathrm{Var}(Q_i K_i) = d_k$$

テキスト テキスト テキスト = $\sqrt{d_k}$. $d_k$テキスト テキスト テキスト テキスト softmaxテキスト **テキスト** (テキスト 値テキスト 1, テキスト 0).

$\sqrt{d_k}$テキスト テキスト テキスト 1テキスト テキスト → softmaxテキスト テキスト 分布 テキスト. テキスト度 テキスト テキスト.


In [ ]:
# sqrt(d_k) テキスト テキスト テキスト
np.random.seed(0)
dks = [8, 64, 512, 2048]
fig, axes = plt.subplots(1, len(dks), figsize=(16, 4))

for ax, d_k in zip(axes, dks):
    n = 4
    Q = np.random.randn(n, d_k)
    K = np.random.randn(n, d_k)
    # テキスト テキスト
    scores_no = Q @ K.T
    # テキスト テキスト
    scores_scaled = scores_no / np.sqrt(d_k)

    # softmax
    p_no = softmax(scores_no)
    p_scaled = softmax(scores_scaled)

    ax.bar(['q0', 'q1', 'q2', 'q3'], p_no[0], alpha=0.5, label='Scaling X', color='red')
    ax.bar(['q0', 'q1', 'q2', 'q3'], p_scaled[0], alpha=0.5, label='Scaling O', color='blue')
    ax.set_title(f'd_k={d_k}\nVariance: {scores_no.var():.1f} → {scores_scaled.var():.1f}')
    ax.set_ylim(0, 1)
    ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig('../figures/ch14_scaling_effect.png', dpi=100, bbox_inches='tight')
plt.show()
print("=> d_kテキスト テキスト Scaling テキストPlane softmaxテキスト テキスト Valueテキスト テキスト (テキスト).")


## 14.4 Attentionテキスト 時間·空間 テキスト度

$$\mathrm{Attn}(Q, K, V) = \mathrm{softmax}(QK^\top / \sqrt{d_k}) V$$

- $QK^\top$: $n \times d \times n = O(n^2 d)$ テキスト
- $\mathrm{softmax}(A) V$: $n \times n \times d = O(n^2 d)$
- **テキスト 時間**: $O(n^2 d)$
- **空間**: $A \in \mathbb{R}^{n \times n}$ → $O(n^2)$

問題: シーケンス長 $n$テキスト テキスト $n^2$テキスト テキスト. 8K テキスト $n^2 = 6.4 \times 10^7$.
→ Flash Attention, テキスト Attention テキスト テキスト (Ch 27テキスト).


In [ ]:
# シーケンス長テキスト テキスト 可視化
seq_lens = [128, 256, 512, 1024, 2048, 4096, 8192]
n2 = [n**2 for n in seq_lens]
n2d = [n**2 * 64 for n in seq_lens]  # d=64

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(range(len(seq_lens)), n2, alpha=0.7, label='$n^2$ (Space)')
ax.bar(range(len(seq_lens)), [n/1000 for n in n2d], alpha=0.7, label='$n^2 \\cdot d$ / 1000 (Operation)')
ax.set_xticks(range(len(seq_lens)))
ax.set_xticklabels([str(n) for n in seq_lens])
ax.set_xlabel('Sequence Length n')
ax.set_ylabel('Complexity')
ax.set_title('Attention Complexity: $O(n^2 d)$ — Sequence Lengthテキスト テキスト テキスト')
ax.legend()
plt.tight_layout()
plt.savefig('../figures/ch14_complexity.png', dpi=100, bbox_inches='tight')
plt.show()
print("8K Contextテキスト 64M テキスト Attention Scores. Flash Attentionテキスト テキスト テキスト.")


## 14.5 テキスト テキスト (Causal Masking)

GPT テキスト autoregressive モデルテキスト テキスト テキスト テキスト テキスト テキスト.

テキスト 行列 $M$:
$$M_{ij} = \begin{cases} 0 & \text{if } i \geq j \text{ (テキスト/テキスト)} \\ -\infty & \text{if } i < j \text{ (テキスト)} \end{cases}$$

$A + M$テキスト softmax → テキスト テキスト テキスト 0テキスト テキスト.


In [ ]:
# テキスト テキスト
n = 6
mask = np.triu(np.ones((n, n)) * -1e9, k=1)  # テキスト テキスト -inf
print("Causal Mask (テキスト テキスト -inf):")
print(mask[:4, :4])

# テキスト テキスト
np.random.seed(0)
scores = np.random.randn(n, n)  # Attention テキスト
print(f"\nOriginal Scores (テキスト テキスト): {scores[0].round(2)}")
masked = scores + mask
print(f"Mask Application (テキスト テキスト): {masked[0].round(2)}")
weights = softmax(masked)
print(f"\nテキスト テキスト (テキスト テキスト): {weights[0].round(4)}")
print(f"  → テキスト テキスト テキスト テキスト テキスト テキスト (1.0)")

# 可視化
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
im0 = axes[0].imshow(scores, cmap='viridis'); plt.colorbar(im0, ax=axes[0])
axes[0].set_title('Original Attention Scores')
im1 = axes[1].imshow(mask != -1e9, cmap='binary'); plt.colorbar(im1, ax=axes[1])
axes[1].set_title('Causal Mask (テキスト=テキスト)')
im2 = axes[2].imshow(weights, cmap='Blues'); plt.colorbar(im2, ax=axes[2])
axes[2].set_title('Mask Application テキスト Weight')
plt.tight_layout()
plt.savefig('../figures/ch14_causal_mask.png', dpi=100, bbox_inches='tight')
plt.show()


## 14.6 [CPU/GPU ベンチマーク ⑤] シーケンス長テキスト Attention 時間

シーケンス長 $n$テキスト 2テキスト テキスト Attention 時間テキスト 4テキスト (テキスト 4テキスト).
CPU vs GPU テキスト テキスト テキスト テキスト.


In [ ]:
# Attention ベンチマーク
import time
from llm_math.bench import time_fn, format_results_table

def attention_torch(Q, K, V, mask=None):
    d_k = Q.shape[-1]
    scores = Q @ K.transpose(-1, -2) / (d_k ** 0.5)
    if mask is not None:
        scores = scores + mask
    weights = F.softmax(scores, dim=-1)
    return weights @ V

# シーケンス長テキスト
print(f"{'n':>6} | {'CPU (ms)':>12} | {'GPU (ms)':>12} | {'Speedup':>10}")
print("-" * 50)
for n in [128, 256, 512, 1024, 2048]:
    d = 64
    Q = torch.randn(n, d)
    K = torch.randn(n, d)
    V = torch.randn(n, d)
    res_cpu = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=3)
    if torch.cuda.is_available():
        Q_g, K_g, V_g = Q.cuda(), K.cuda(), V.cuda()
        res_gpu = time_fn(attention_torch, Q_g, K_g, V_g, device='cuda', warmup=3, repeat=5)
        speedup = res_cpu['mean_ms'] / res_gpu['mean_ms']
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {res_gpu['mean_ms']:>12.3f} | {speedup:>9.2f}x")
    else:
        print(f"{n:>6} | {res_cpu['mean_ms']:>12.3f} | {'N/A':>12} | {'N/A':>10}")

if not torch.cuda.is_available():
    print("\n⚠ GPU テキスト. n=2048 テキスト度Plane CPUテキスト テキスト テキスト テキスト.")


In [ ]:
# PyTorch SDPA (scaled_dot_product_attention) テキスト
print("PyTorch SDPA Function テキスト:")
print("(テキスト Flash Attention テキスト Optimization テキスト テキスト Lineテキスト)\n")

n, d = 512, 64
Q = torch.randn(1, n, d)
K = torch.randn(1, n, d)
V = torch.randn(1, n, d)

# テキスト テキスト
out_manual = attention_torch(Q, K, V)

# PyTorch SDPA (F.scaled_dot_product_attention)
out_sdpa = F.scaled_dot_product_attention(Q, K, V)

print(f"テキスト Implementation vs SDPA Maximum Error: {(out_manual - out_sdpa).abs().max():.2e}")
print("\n=> テキスト Result. SDPAテキスト テキスト Optimizationテキスト テキスト.")

# 速度 比較
print("\nSpeed Comparison (n=512):")
t_manual = time_fn(attention_torch, Q, K, V, device='cpu', warmup=2, repeat=5)
def sdpa_call(Q, K, V):
    return F.scaled_dot_product_attention(Q, K, V)
t_sdpa = time_fn(sdpa_call, Q, K, V, device='cpu', warmup=2, repeat=5)
print(f"  テキスト Implementation: {t_manual['mean_ms']:.3f} ms")
print(f"  PyTorch SDPA: {t_sdpa['mean_ms']:.3f} ms")
print(f"  Speed Improvement: {t_manual['mean_ms'] / t_sdpa['mean_ms']:.2f}x")


## 14.7 Attention バックプロパゲーション (テキスト)

テキスト テキスト テキスト テキスト テキスト. テキスト:

$$\frac{\partial \mathcal{L}}{\partial Q} = \frac{1}{\sqrt{d_k}} \left(\frac{\partial \mathcal{L}}{\partial A} V K^\top + \ldots\right)$$

PyTorchテキスト `autograd`テキスト テキスト テキスト. Flash Attentionテキスト テキスト バックプロパゲーションテキスト テキスト テキスト (Ch 27).

## 14.8 要点

| テキスト | テキスト | テキスト |
|---|---|---|
| Attention | $\mathrm{softmax}(QK^\top/\sqrt{d_k})V$ | テキスト-テキスト テキスト度テキスト 値 テキスト |
| テキスト | $\frac{1}{\sqrt{d_k}}$ | テキスト 1テキスト テキスト |
| テキスト テキスト | $M_{ij} = -\infty$ if $i < j$ | テキスト テキスト テキスト |
| テキスト度 | $O(n^2 d)$ 時間, $O(n^2)$ 空間 | シーケンス長テキスト 2テキスト |

## 演習問題

1. $\mathrm{Var}(QK^\top_{ij}) = d_k$テキスト Q, Kテキスト テキスト iid $\mathcal{N}(0,1)$テキスト テキスト テキスト度テキスト.
2. テキスト テキスト テキスト Attentionテキスト テキスト Attentionテキスト 出力テキスト 比較テキスト.
3. Attention テキスト 行列テキスト 可視化テキスト, テキスト テキスト テキスト テキスト テキスト.
4. シーケンス長 256, 512, 1024, 2048テキスト CPU 時間テキスト テキスト $O(n^2)$テキスト 検証テキスト.
5. PyTorch SDPAテキスト `is_causal=True` テキスト テキスト テキスト Attentionテキスト テキスト.

> 解答: `solutions/ch14_solutions.ipynb`
